article: https://arxiv.org/abs/2608.22274

# Окружение

для скорости (и чтобы Llada-8B поместилась) будем использовать 2 x GPU T4 одновременно

для этого потребовалась тонкая настройка совместимостей

In [ ]:
%pip -q install "transformers==4.51.3" "peft==0.15.2" "accelerate==1.7.0" "bitsandbytes==0.50.2" "huggingface_hub>=0.30,<1.0" "sacrebleu>=2.4,<3"

In [1]:
import json
import random
import subprocess
import sys
import time
from pathlib import Path

import torch
import pandas as pd
import sacrebleu
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

I0000 00:00:1794909839.838461     114 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


лора адаптер и датасет для теста возьмем из официального [репозитория](https://github.com/Entropy-Valley/Entropy-Valley) для более качественной имплементации

In [2]:
ROOT = Path("/kaggle/working/Entropy-Valley")
if not (ROOT / "ladit").exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Entropy-Valley/Entropy-Valley.git", str(ROOT)], check=True)
sys.path.insert(0, str(ROOT))

from ladit.data.mt_dataset import set_lang_pair
from ladit.decoding.length_adaptive import entropy_valley_probe, set_mask_token_id as set_ev_mask
from ladit.decoding.translate import translate_single, set_mask_token_id as set_dec_mask
from ladit.model.backbone import LLADA_MASK_TOKEN_ID

Cloning into '/kaggle/working/Entropy-Valley'...


In [3]:
BASE = "GSAI-ML/LLaDA-8B-Base"
ADAPTER = "YanZhanPKU/Entropy-Valley-LLaDA-8B-En2Zh"
DATA_REPO = "YanZhanPKU/Entropy-Valley-Datasets"

path = hf_hub_download(DATA_REPO, "wmt22_enzh_test.jsonl", repo_type="dataset")
with open(path, encoding="utf-8") as f:
    rows = [json.loads(line) for line in f if line.strip()]
print(len(rows))

wmt22_enzh_test.jsonl: 0.00B [00:00, ?B/s]

2037


In [4]:
# настраиваем свой девайс мап чтобы все завелось на двух гпу одновре
device_map = {
    "model.transformer.wte": 0,
    "model.transformer.emb_drop": 0,
}
for i in range(32):
    device_map[f"model.transformer.blocks.{i}"] = 0 if i < 16 else 1

device_map["model.transformer.ln_f"] = 1
device_map["model.transformer.ff_out"] = 1

quant = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    BASE,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    quantization_config=quant,
    device_map=device_map,
    low_cpu_mem_usage=True,
)
model = PeftModel.from_pretrained(model, ADAPTER, is_trainable=False)
model.eval()

# выбираем пару АНГЛ-КИТАЙ
set_lang_pair("en-zh")
mask_id = int(getattr(AutoConfig.from_pretrained(BASE, trust_remote_code=True),
                      "mask_token_id", LLADA_MASK_TOKEN_ID))
set_ev_mask(mask_id)
set_dec_mask(mask_id)
DEVICE = str(model.get_input_embeddings().weight.device)

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/766 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

configuration_llada.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/GSAI-ML/LLaDA-8B-Base:
- configuration_llada.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


... (11 outputs left out)

generation_config.json:   0%|          | 0.00/128 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

/usr/local/lib/python3.13/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['qalora_group_size', 'target_parameters', 'use_qalora'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


adapter_model.safetensors:   0%|          | 0.00/637M [00:00<?, ?B/s]

# Эксперименты

In [5]:
N = 400
T = 32
SEED = 67
RATIOS = [0.70, 0.75, 0.80, 0.85, 0.90]
RATIO_BASELINE = 0.8 # из статьи
OUTPUT = Path("/kaggle/working/ev_400_results.jsonl")

eligible = []
for idx, row in enumerate(rows):
    if not row.get("en") or not row.get("zh"):
        continue
    src_len = len(tokenizer.encode(row["en"], add_special_tokens=False))
    ref_len = len(tokenizer.encode(row["zh"], add_special_tokens=False))
    
    # выберем только не большие предложения на перевод
    if 8 <= src_len <= 30 and 2 <= ref_len <= 50:
        eligible.append({
            "id": idx, "source": row["en"], "reference": row["zh"],
            "src_len": src_len, "ref_len": ref_len,
        })

selected = sorted(random.Random(SEED).sample(eligible, N), key=lambda r: r["id"])

In [6]:
completed = {}
if OUTPUT.exists():
    with OUTPUT.open(encoding="utf-8") as f:
        for line in f:
            if line.strip():
                item = json.loads(line)
                completed[item["id"]] = item

with torch.inference_mode():
    for row in tqdm(selected):
        if row["id"] in completed:
            continue

        n = row["src_len"]
        ratio_len = max(1, int(n * RATIO_BASELINE))
        oracle_len = row["ref_len"] + 1
        candidates = sorted({max(1, int(n * r)) + 1 for r in RATIOS})
        probe = entropy_valley_probe(model, tokenizer, row["source"], candidates, device=DEVICE)
        ev_len = int(probe["best_length"])

        translations = {}
        lengths = {"ratio": ratio_len, "ev": ev_len, "oracle": oracle_len}
        for method, length in lengths.items():
            result = translate_single(
                model, tokenizer, row["source"],
                target_length=length, num_steps=T,
                schedule_name="med", temperature=0.0, device=DEVICE,
            )
            translations[method] = result["translation"]

        item = {
            **row,
            "lengths": lengths,
            "translations": translations,
            "ev_entropies": probe["results"],
        }
        with OUTPUT.open("a", encoding="utf-8") as f:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")
        completed[row["id"]] = item

  0%|          | 0/400 [00:00<?, ?it/s]

# Результаты

In [10]:
# только выбранные 400 записей в фиксированном порядке.
results = [completed[row["id"]] for row in selected]
refs = [r["reference"] for r in results]
ref_lengths = [r["ref_len"] + 1 for r in results]

summary = []
for method in ["ratio", "ev", "oracle"]:
    preds = [r["translations"][method] for r in results]
    lengths = [r["lengths"][method] for r in results]
    summary.append({
        "Method": method,
        "sacreBLEU": sacrebleu.corpus_bleu(preds, [refs], tokenize="zh").score,
        "chrF": sacrebleu.corpus_chrf(preds, [refs]).score,
        "Length MAE": sum(abs(x-y) for x,y in zip(lengths,ref_lengths)) / N
    })

table = pd.DataFrame(summary).set_index("Method").round(3)
table.to_csv("/kaggle/working/ev_400_metrics.csv")
table

,sacreBLEU,chrF,Length MAE
Method,,,
ratio,34.797,30.671,2.652
ev,36.261,32.446,2.125
oracle,39.771,35.363,0.000


In [9]:
# несколько примеров
for r in results[:5]:
    print("\nEN:", r["source"])
    print("Референс:", r["reference"])
    for method in ["ratio", "ev", "oracle"]:
        print(method, "L=", r["lengths"][method], r["translations"][method])


EN: I'm sorry that your order is running late.
Референс: 很抱歉，您点的餐可能会晚到一会。
ratio L= 8 很抱歉，您的订单来晚了。
ev L= 8 很抱歉，您的订单来晚了。
oracle L= 11 很抱歉,您的订餐被耽搁了.

EN: Please note that if you pay for expedited shipping, the handling time for the order is still 3 business days.
Референс: 请注意，即使您支付了加急运输的费用，处理订单的时间仍然是 3 个工作日。
ratio L= 19 请注意：如果您付了加急运邮费，订单的处理时间仍然是三个工作日。
ev L= 17 请注意即使您付了快速邮费，订单的处理时间仍然是三个工作日。
oracle L= 19 请注意：如果您付了加急运邮费，订单的处理时间仍然是三个工作日。

EN: Whenever the item ships, it will be shipped by the expedited service if you have paid for it.
Референс: 不过，如果您支付了此费用，无论何时发货，都将通过加急服务运输。
ratio L= 16 如果你支付了快递服务，无论何时发货，它就会通过快递服务发货。
ev L= 15 只要您已付款，无论何时发货，它就会通过快速服务发货。
oracle L= 17 如果你已经支付了费用，那么无论何时发货，它就会通过快递服务发货。

EN: International shipments are usually delivered within 11 to 22 business days, depending on the time it takes to clear customs.
Референс: 国际包裹通常在 11 至 22 个工作日内送达，具体时间取决于清关时间。
ratio L= 20 国际运输通常会在11到22个营业日内送达，这取决于通过海关的时间。
ev L= 19 国际运输通常会在11到22个工作日内送达，这取决于海关通关的时间。
oracle L= 22 国际空运通常会在11到22个营业日